In [1]:
!pip install bertviz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.5/157.5 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.3/139.3 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.0/14.0 MB 74.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.7/85.7 kB 7.9 MB/s eta 0:00:00


In [2]:
from bertviz.neuron_view import show
from bertviz.transformers_neuron_view import BertModel
from transformers import AutoTokenizer, AutoConfig

model_ckpt = 'bert-base-uncased'
model = BertModel.from_pretrained(model_ckpt)
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
text = 'As the aircraft becomes lighter, it flies higher in air of lower density to maintain the same airspeed'
show(model, 'bert', tokenizer, text, display_mode='light', layer=0, head=8)

Output hidden; open in https://colab.research.google.com to view.

In [4]:
inputs = tokenizer(text, return_tensors='pt', add_special_tokens=False)
inputs.input_ids

tensor([[ 2004,  1996,  2948,  4150,  9442,  1010,  2009, 10029,  3020,  1999,
          2250,  1997,  2896,  4304,  2000,  5441,  1996,  2168, 14369, 25599]])

In [3]:
config = AutoConfig.from_pretrained(model_ckpt)
config

BertConfig {
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.56.1",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

In [5]:
import torch
import torch.nn as nn

token_emb = nn.Embedding(config.vocab_size, config.hidden_size)
token_emb

Embedding(30522, 768)

In [6]:
input_embs = token_emb(inputs.input_ids)
print(input_embs)
input_embs.shape

tensor([[[ 0.2041,  0.1073,  0.2806,  ..., -1.0565,  0.3174, -0.6988],
         [ 0.1979, -0.6360, -0.9787,  ...,  0.2536, -1.1333,  0.5469],
         [-0.2479,  0.3648,  0.3135,  ..., -1.0942,  0.3676, -1.1586],
         ...,
         [-1.5001, -0.8872,  1.2150,  ...,  1.5024, -0.3859, -0.5033],
         [-1.1077,  0.2474, -0.0843,  ...,  0.4318,  0.2639,  1.8803],
         [ 0.1962,  1.2372,  1.2849,  ..., -0.3568, -1.1091, -1.5800]]],
       grad_fn=<EmbeddingBackward0>)


torch.Size([1, 20, 768])

In [7]:
# Creating query, key, value vectors and calculating the attention scores
import math
import torch.nn.functional as f
#query = key = value = input_embs

In [8]:
def scaled_dot_product_attention(query, key, value):
  dim_k = key.size(-1)
  scores = torch.bmm(query, key.transpose(1, 2)) / math.sqrt(dim_k)
  weights = f.softmax(scores, dim=1)
  attn_outputs = torch.bmm(weights, value)
  return attn_outputs # contextual embeddings generated from general embeddings

In [9]:
class AttentionHead(nn.Module):
  def __init__(self, embed_dim, head_dim):
    super(AttentionHead, self).__init__()
    self.q = nn.Linear(embed_dim, head_dim)
    self.k = nn.Linear(embed_dim, head_dim)
    self.v = nn.Linear(embed_dim, head_dim)
  def forward(self, hidden_state):
    attn_outputs = scaled_dot_product_attention(self.q(hidden_state), self.k(hidden_state), self.v(hidden_state))
    return attn_outputs

In [10]:
class MultiHeadAttention(nn.Module):
  def __init__(self, config):
    super().__init__()
    embed_dim = config.hidden_size
    num_heads = config.num_attention_heads
    head_dim = embed_dim // num_heads # No. of multiple heads
    self.heads = nn.ModuleList(
        [AttentionHead(embed_dim, embed_dim)]
    )
    self.output_linear = nn.Linear(embed_dim, embed_dim)
  def forward(self, hidden_state):
    concatenated_output = torch.concat([h(hidden_state) for h in self.heads])
    multi_head_output = self.output_linear(concatenated_output)
    return multi_head_output

In [11]:
multiHead = MultiHeadAttention(config)
attn_op = multiHead(input_embs)
attn_op.shape

torch.Size([1, 20, 768])

In [20]:
from bertviz import head_view
from transformers import AutoModel
from IPython.display import display

model1 = AutoModel.from_pretrained(model_ckpt, output_attentions=True)
sent_a = text = 'As the aircraft becomes lighter, it flies higher in air of lower density to maintain the same airspeed'
sent_b = 'the corn field is full of flies'
input_tensor = tokenizer(sent_a, sent_b, return_tensors='pt')
attn = model1(**input_tensor).attentions
sent_b_start = (input_tensor.token_type_ids == 0).sum(dim=1)
tokens = tokenizer.convert_ids_to_tokens(input_tensor.input_ids[0])
display(head_view(attn, tokens, sentence_b_start=sent_b_start, heads=[21]))

Output hidden; open in https://colab.research.google.com to view.

In [26]:
class FeedFwdNN(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.linear1 = nn.Linear(config.hidden_size, config.intermediate_size)
    self.linear2 = nn.Linear(config.intermediate_size, config.hidden_size)
    self.gelu = nn.GELU()
    self.dropout = nn.Dropout(config.hidden_dropout_prob)
  def forward(self, x):
    x = self.linear1(x)
    x = self.gelu(x)
    x = self.linear2(x)
    x = self.dropout(x)
    return x

In [28]:
feed_fwd = FeedFwdNN(config)
ff_op = feed_fwd(attn_op)
ff_op

tensor([[[-0.0083, -0.0123,  0.0108,  ...,  0.0035, -0.0189, -0.0225],
         [-0.0125, -0.0000,  0.0117,  ..., -0.0053, -0.0139, -0.0208],
         [-0.0076, -0.0116,  0.0000,  ..., -0.0102, -0.0268, -0.0296],
         ...,
         [-0.0125, -0.0119,  0.0061,  ..., -0.0020, -0.0184, -0.0239],
         [-0.0079, -0.0034,  0.0070,  ...,  0.0039, -0.0304, -0.0000],
         [-0.0116, -0.0124,  0.0090,  ...,  0.0010, -0.0241, -0.0240]]],
       grad_fn=<MulBackward0>)

In [32]:
class Encoder(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.layer_norm_1 = nn.LayerNorm(config.hidden_size)
    self.layer_norm_2 = nn.LayerNorm(config.hidden_size)
    self.attention = MultiHeadAttention(config)
    self.FeedForward = FeedFwdNN(config)

  def forward(self, x):
    hidden_state = self.layer_norm_1(x)
    x = x + self.attention(hidden_state)
    x = x + self.FeedForward(self.layer_norm_2(x))
    return x

In [33]:
encoder = Encoder(config)
input_embs.shape, encoder(input_embs).shape

(torch.Size([1, 20, 768]), torch.Size([1, 20, 768]))

In [42]:
class PositionalEncoding(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.token_embeddings = nn.Embedding(config.vocab_size, config.hidden_size)
    self.positional_embeddings = nn.Embedding(config.max_position_embeddings, config.hidden_size)
    self.layer_norm = nn.LayerNorm(config.hidden_size)
    self.dropout = nn.Dropout()

  def forward(self, input_ids):
    seq_length = input_ids.size(1)
    pos_id = torch.arange(seq_length, dtype=torch.long)

    token_embeddings = self.token_embeddings(input_ids)
    position_embeddings = self.positional_embeddings(pos_id)

    embedding_final = token_embeddings + position_embeddings
    embedding_final = self.layer_norm(embedding_final)
    embedding_final = self.dropout(embedding_final)
    return embedding_final

In [46]:
positional_encoding_layer = PositionalEncoding(config)
op = positional_encoding_layer(inputs.input_ids)
op.size()

torch.Size([1, 20, 768])

In [49]:
class TransformerEncode(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.positional_encoding = PositionalEncoding(config)
    self.layers = nn.ModuleList([Encoder(config) for _ in range(config.num_hidden_layers)])

  def forward(self, x):
    x = self.positional_encoding(x)
    for layer in self.layers:
      x = layer(x)
      return x

In [50]:
enc = TransformerEncode(config)
enc(inputs.input_ids).shape

torch.Size([1, 20, 768])